In [ ]:
"""
Q- Find the highest salary from a table 

select employee_name, max(salary) from employee_table

"""

In [ ]:
## Second highest salary  
"""
## approach 1 by using limit and subquery (do not use this approch in interview)

select salary from (slect salary from employee_table
order by salary desc limit 2) as temp 
order by salary asc limit 1

"""

## approch(2) 
"""
select max(salary) from employee_table  where salary < (
select max(salary) from employee_table)
"""

## approach(3) by using the window function dens_rank()

"""

select salary from 
 (select salary, dense_rank() over(order by salary desc) as d_rnk
 from employee_table) as tem_result
 where d_rank=2;
 

"""


In [ ]:
## Highest salary per department

"""
approach 1 
 select department_name, max(salary) as highest_salary
 from employee_table group by department_name

"""
## approach 2 
"""
 select  salary, employee_name, department_name from
 (select salary, department_name, employee_name, rank() over (partition by department_name order by salary desc) as max_salary 
 FROM employee_table ) tmp
 where max_salary=1

"""


In [ ]:
## second highest salary per department 

"""
## approach1 using group by 

-- second highest salary 
select department_name, max(salary) as salary from employee_table e1
where salary < (
select max(salary) from employee_table d1 where d1.department_name=e1.department_name)
group by department_name

 
##byusing windows function  (approach) 2

select department_name, employee_name, salary from 
(select department_name, employee_name, salary , dense_rank() over(partition by department_name order by salary desc) as drnk
from employee_table) t 
where drnk=2;

"""


In [ ]:
## Find total salary paid per department
"""
## approach 1 

select department, sum(salary) as total_salary  
from employee_table group by department

## approach 2 using window_function  -
select distinct department_name, total_salary from (
select department_name, sum(salary) over(partition by department_name) as total_salary
from department_table) tmp;

Note- why distinct here because (windows function here produced the multiple rows per department 
so we used distinct)

"""

In [ ]:
##	Find departments where avg salary > 60,000

"""  
## approach1  (best approach)
select department_name, avg(salary) as avg_salary
from employee_table
group by department_name 
having avg(salary)>600000   -- we can use having clause 


## approach 2 using window function
select DISTINCT deparment_name, avg_salary from 
  (selet department_name, avg(salary) over (partition by department) as avg_salary  
  from employee_table) tmp 
  where avg_salary>60000


"""

In [ ]:
## Find duplicate employee names
"""
## approach1  ( this approach is good for summary ) count(*) does not ignore nulls

 select employee_name, count(employee_name) as cnt from employee_table
  group by employee_name
  having count(employee_name)>1;  


## by using window function  ( for large table scan )

  select * from (
  select employee_name, count(employee_name) over(partition by employee_name) as cnt 
  from employee_table ) tmp 
  where cnt>1;

"""


## ------------What if interviewer asks:

# “Show me the actual duplicate rows, not just the names”

"""
Answer -  ( for large table scan)
select * from (
 select employee_name
 from employee_table 
 group by employee_name 
 having count(employee_name) >1);

"""

In [ ]:
##Remove duplicate employee records, keeping only one row per name

## approach 1 keep the row with first employee_id (lowest employee id )
"""
with duplicate_rows as (
select *, row_number() over(partion by employee_name) as rn 
from employee_table 
)

delete from  employee_table where employee_id in (
select employee_id from duplicate_rows where
rn>1);


----------approach 2 (keep only last updated rows)

with duplicate_rows (
select *, row_number() over(partition by employee_name order by ts desc) as rn  from employee_table
)
delete from employee_table where employee_id in
(select employee_id from duplicate_rows where rn>1);


"""


In [ ]:
## 10.	Find employees earning more than average salary

"""  
## approach 1 (scaler subquery- the subquery runs only once)

select employee_name from employee_table where salary > (
 select avg(salary) from employee_table 
);

## approach 2 (window function) (without subquery)
select  employee_name, salary from (
select employee_name, salary, avg(salary) over () as avg_salary from employee_table ) t 
where salary> avg_salary; 


"""


In [ ]:
## 11.	Find departments with no employees

"""  both approaches are wrong ,because group by cannot be follow on now rows departments
approach 1 

select department, count(employee_id) as employee_cnt from employee_table
group by department
having count(employee_id)<1;

--approach 2


select department_name, ep_cnt (
select department_name, count(employee_id) over (partition by departemnt_name) as ep_cnt
 from employee_table
 )
 where ep_cnt<1;

"""
# -------------correct way
    #LEFT JOIN keeps all departments

   #Departments with no employees → NULL on employee side
   #Filter NULLs → exactly what we want
"""
## left join 
select d.department_name from department d
left join employe_table e
on d.department_id=e.department_id
where e.employee_id is null;

## approach 2

select d.department_name from department_table d
where not exists (
select 1 from employee_table 
where e.department_id=d.department_id);
"""
